# Fase 3: Preparación de datos
## IISE Challenge 2026 — Optimización de Combustible

Objetivos:
1. Limpiar datos (duplicados, outliers, formatos inconsistentes)
2. Crear variables derivadas (duración, velocidad, día de semana)
3. Exportar dataset limpio a `data/processed/`

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import time

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

df = pd.read_excel("../data/raw/BaseDatos_IISE_TO.xlsx")
print(f"Shape original: {df.shape}")
df.head()

Shape original: (75, 12)


,Remisión,Unidad,Operador,Ruta,Fecha Inicio (Día),Fecha Fin (Día),Inicio (Hora),Fin (Hora),km,Litros,Rend,peso
0,P-110460,3157,DESGARENNIS ORTEGA CHRISTIAN,PUEBLA -CUAUTITLAN,2026-03-23,2026-03-23,04:59:00,16:46:00,346.4,137.0,2.528467,35.995
1,P-109685,3157,DESGARENNIS ORTEGA CHRISTIAN,PUEBLA -CUAUTITLAN,2026-02-18,2026-02-18,04:48:00,21:01:00,386.8,155.5,2.487460,34.761
2,P-109704,3077,APANECATL MORENO SAUL,PUEBLA -CUAUTITLAN,2026-02-18,2026-02-18,04:58:00,20:20:00,385.2,142.0,2.712676,34.778
3,P-109662,3158,PASTRANA CARIÑO FRANCISCO SAMUEL,PUEBLA -CUAUTITLAN,2026-02-17,2026-02-17,05:00:00,23:11:00,350.4,149.5,2.343813,34.803
4,P-109576,3158,PASTRANA CARIÑO FRANCISCO SAMUEL,PUEBLA -CUAUTITLAN,2026-02-16,2026-02-16,04:58:00,22:03:00,341.6,156.5,2.182748,34.828


## 1. Investigar remisión duplicada
Hay 74 remisiones únicas en 75 filas — encontrar el duplicado.

In [3]:
# Encontrar la remisión duplicada
dupes = df[df.duplicated(subset='Remisión', keep=False)]
print(f"Remisión duplicada: {dupes['Remisión'].iloc[0]}")
dupes

Remisión duplicada: P-108970


,Remisión,Unidad,Operador,Ruta,Fecha Inicio (Día),Fecha Fin (Día),Inicio (Hora),Fin (Hora),km,Litros,Rend,peso
43,P-108970,3077,APANECATL MORENO SAUL,PUEBLA -CUAUTITLAN,2026-01-27,2026-01-27,05:04:00 a. m.,06:59:00 p. m.,355.4,131.0,2.712977,34.634
50,P-108970,3077,APANECATL MORENO SAUL,PUEBLA -CUAUTITLAN,2026-01-26,2026-01-26,05:03:00 a. m.,05:31:00 p. m.,355.4,134.0,2.652239,34.852


### Decisión: Remisión duplicada P-108970
Son dos viajes distintos (fechas diferentes, litros y peso diferentes) con el 
mismo ID de remisión — error de captura. Se conservan ambos registros ya que 
representan viajes reales. Remisión no se usará como feature en el modelo.

In [4]:
# Investigar outliers
print("=== OUTLIERS A INVESTIGAR ===\n")

print("1. Viaje de máxima distancia:")
print(df[df['km'] == df['km'].max()][['Operador', 'km', 'Litros', 'Rend', 'peso', 'Fecha Inicio (Día)']].to_string())

print("\n2. Viaje de máximo consumo:")
print(df[df['Litros'] == df['Litros'].max()][['Operador', 'km', 'Litros', 'Rend', 'peso', 'Fecha Inicio (Día)']].to_string())

print("\n3. Viaje con mejor rendimiento:")
print(df[df['Rend'] == df['Rend'].max()][['Operador', 'km', 'Litros', 'Rend', 'peso', 'Fecha Inicio (Día)']].to_string())

print("\n4. Viaje con peor rendimiento:")
print(df[df['Rend'] == df['Rend'].min()][['Operador', 'km', 'Litros', 'Rend', 'peso', 'Fecha Inicio (Día)']].to_string())

=== OUTLIERS A INVESTIGAR ===

1. Viaje de máxima distancia:
                      Operador     km  Litros      Rend    peso Fecha Inicio (Día)
72  ARIAS MENDOZA LUIS ENRIQUE  423.6   132.0  3.209091  34.799         2026-01-15

2. Viaje de máximo consumo:
                    Operador     km  Litros      Rend    peso Fecha Inicio (Día)
47  BELLO CORONA JUAN CARLOS  377.1   185.5  2.032884  34.866         2026-01-23

3. Viaje con mejor rendimiento:
                      Operador     km  Litros      Rend    peso Fecha Inicio (Día)
72  ARIAS MENDOZA LUIS ENRIQUE  423.6   132.0  3.209091  34.799         2026-01-15

4. Viaje con peor rendimiento:
                    Operador     km  Litros      Rend    peso Fecha Inicio (Día)
47  BELLO CORONA JUAN CARLOS  377.1   185.5  2.032884  34.866         2026-01-23


### Decisión: Outliers

**Fila 72 (Arias, 423.6 km, Rend 3.21):** Distancia atípica (+100 km) pero 
el mejor rendimiento del dataset. Podría ser ruta alterna o error de captura 
en km. Se conserva pero se marca como outlier.

**Fila 47 (Bello Corona, 185.5 L, Rend 2.03):** Consumo extremo con distancia 
normal. Posible ralentí excesivo, falla mecánica, o error de captura. Se 
conserva pero se marca como outlier.

Con solo 75 registros, eliminar datos es costoso. Se marcan para poder 
analizar el modelo con y sin ellos.

In [5]:
# Marcar outliers en vez de eliminar
df['outlier'] = False
df.loc[df['km'] == df['km'].max(), 'outlier'] = True      # 423.6 km
df.loc[df['Litros'] == df['Litros'].max(), 'outlier'] = True  # 185.5 L

print(f"Outliers marcados: {df['outlier'].sum()}")
print(f"Registros limpios: {(~df['outlier']).sum()}")

Outliers marcados: 2
Registros limpios: 73


## 2. Feature Engineering — Variables derivadas

In [6]:
from datetime import time

# Función para convertir horas mixtas (datetime.time + strings AM/PM español)
def to_decimal_hour(t):
    if isinstance(t, time):
        return t.hour + t.minute / 60 + t.second / 3600
    t = t.strip()
    es_pm = 'p. m.' in t.lower() or 'p.m.' in t.lower()
    hora_str = t.split('a. m.')[0].split('p. m.')[0].split('a.m.')[0].split('p.m.')[0].strip()
    parts = hora_str.split(':')
    h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
    if es_pm and h != 12:
        h += 12
    elif not es_pm and h == 12:
        h = 0
    return h + m / 60 + s / 3600

# --- Variables derivadas ---
# 1. Horas decimales
df['hora_inicio'] = df['Inicio (Hora)'].apply(to_decimal_hour)
df['hora_fin'] = df['Fin (Hora)'].apply(to_decimal_hour)

# 2. Duración del viaje (horas)
df['duracion_hrs'] = df['hora_fin'] - df['hora_inicio']
df.loc[df['duracion_hrs'] < 0, 'duracion_hrs'] += 24  # cruces de medianoche

# 3. Velocidad promedio (km/h)
df['vel_promedio'] = df['km'] / df['duracion_hrs']

# 4. Día de la semana
df['dia_semana'] = df['Fecha Inicio (Día)'].dt.day_name()

# 5. Costo del viaje en pesos ($28.20/L)
df['costo_combustible'] = df['Litros'] * 28.20

print("Variables creadas:")
print(df[['hora_inicio', 'hora_fin', 'duracion_hrs', 'vel_promedio', 'dia_semana', 'costo_combustible']].describe().round(2))

Variables creadas:
       hora_inicio  hora_fin  duracion_hrs  vel_promedio  costo_combustible
count        75.00     75.00         75.00         75.00              75.00
mean          5.22     17.62         13.04         34.16            4023.76
std           1.86      4.33          3.99         46.80             277.30
min           4.02      4.80          0.78         14.38            3496.80
25%           4.97     15.18         10.22         21.30            3842.25
50%           5.00     18.12         13.08         26.97            3976.20
75%           5.05     21.62         16.86         33.71            4166.55
max          21.02     23.90         23.90        426.38            5231.10


In [7]:
# Viajes con velocidad imposible o duración sospechosa
print("Viajes con velocidad > 100 km/h (imposible para tractocamión):")
sospechosos = df[df['vel_promedio'] > 100][['Operador', 'km', 'duracion_hrs', 'vel_promedio', 'hora_inicio', 'hora_fin', 'Fecha Inicio (Día)', 'Fecha Fin (Día)']]
print(sospechosos.to_string())

Viajes con velocidad > 100 km/h (imposible para tractocamión):
                    Operador     km  duracion_hrs  vel_promedio  hora_inicio  hora_fin Fecha Inicio (Día) Fecha Fin (Día)
49  ARELLANO ALBERTO IGNACIO  334.0      0.783333    426.382979         4.05  4.833333         2026-01-24      2026-01-24


### Viaje sospechoso: Fila 49 (Arellano, vel = 426 km/h)
Hora fin registrada como ~4:50 AM, probablemente es 4:50 PM (error de 
formato AM/PM). No podemos corregirlo con certeza, así que se marca la 
duración y velocidad de este viaje como no confiables.

In [8]:
# Marcar viaje con duración no confiable
df.loc[49, 'outlier'] = True

print(f"Outliers totales: {df['outlier'].sum()}")
print(f"Registros limpios: {(~df['outlier']).sum()}")

Outliers totales: 3
Registros limpios: 72


In [11]:
# Verificar columnas finales
print("Columnas del dataset procesado:")
print(df.columns.tolist())
print(f"\nShape: {df.shape}")
df.head()

Columnas del dataset procesado:
['Remisión', 'Unidad', 'Operador', 'Ruta', 'Fecha Inicio (Día)', 'Fecha Fin (Día)', 'Inicio (Hora)', 'Fin (Hora)', 'km', 'Litros', 'Rend', 'peso', 'outlier', 'hora_inicio', 'hora_fin', 'duracion_hrs', 'vel_promedio', 'dia_semana', 'costo_combustible']

Shape: (75, 19)


,Remisión,Unidad,Operador,Ruta,Fecha Inicio (Día),Fecha Fin (Día),Inicio (Hora),Fin (Hora),km,Litros,Rend,peso,outlier,hora_inicio,hora_fin,duracion_hrs,vel_promedio,dia_semana,costo_combustible
0,P-110460,3157,DESGARENNIS ORTEGA CHRISTIAN,PUEBLA -CUAUTITLAN,2026-03-23,2026-03-23,04:59:00,16:46:00,346.4,137.0,2.528467,35.995,False,4.983333,16.766667,11.783333,29.397454,Monday,3863.4
1,P-109685,3157,DESGARENNIS ORTEGA CHRISTIAN,PUEBLA -CUAUTITLAN,2026-02-18,2026-02-18,04:48:00,21:01:00,386.8,155.5,2.487460,34.761,False,4.800000,21.016667,16.216667,23.852004,Wednesday,4385.1
2,P-109704,3077,APANECATL MORENO SAUL,PUEBLA -CUAUTITLAN,2026-02-18,2026-02-18,04:58:00,20:20:00,385.2,142.0,2.712676,34.778,False,4.966667,20.333333,15.366667,25.067245,Wednesday,4004.4
3,P-109662,3158,PASTRANA CARIÑO FRANCISCO SAMUEL,PUEBLA -CUAUTITLAN,2026-02-17,2026-02-17,05:00:00,23:11:00,350.4,149.5,2.343813,34.803,False,5.000000,23.183333,18.183333,19.270394,Tuesday,4215.9
4,P-109576,3158,PASTRANA CARIÑO FRANCISCO SAMUEL,PUEBLA -CUAUTITLAN,2026-02-16,2026-02-16,04:58:00,22:03:00,341.6,156.5,2.182748,34.828,False,4.966667,22.050000,17.083333,19.996098,Monday,4413.3


In [12]:
# Exportar a data/processed/
df.to_csv("../data/processed/datos_limpios.csv", index=False)
print("Dataset guardado en data/processed/datos_limpios.csv")

Dataset guardado en data/processed/datos_limpios.csv
